# Step -1: Business Problem Understanding

#### Develop a language translator which converts telugu text to roman telugu and vice versa.

# Step -2: Data Collection

In [1]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
df1 = pd.read_parquet('/home/cs24m118/practice/datasets/project1/train-00000-of-00002.parquet')
df2 = pd.read_parquet('/home/cs24m118/practice/datasets/project1/train-00001-of-00002.parquet')
df = pd.concat([df1, df2], axis = 0)

# Step -3: Data Understanding

In [3]:
print(df.describe())

                                                     text  \
count                                               47415   
unique                                              46736   
top                                                   ...   
freq                                                    2   

                                                 translit  
count                                               47415  
unique                                              46763  
top                                                   ...  
freq                                                    2  


In [4]:
print(df.info())

<class 'pandas.DataFrame'>
Index: 47415 entries, 0 to 23706
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   text      47415 non-null  str  
 1   translit  47415 non-null  str  
dtypes: str(2)
memory usage: 528.9 MB
None


In [5]:
print(df.shape)

(47415, 2)


In [6]:
print(df.isnull().sum())

text        0
translit    0
dtype: int64


In [7]:
print(df.duplicated().sum())

652


In [8]:
print(df.columns.to_list())

['text', 'translit']


In [9]:
print(df.dtypes)

text        str
translit    str
dtype: object


In [10]:
sample = df.sample(1).iloc[0]

print("ROMAN TELUGU:\n")
print(sample["translit"])

print("\n" + "=" * 80 + "\n")

print("TELUGU:\n")
print(sample["text"])

ROMAN TELUGU:


rashtramlo prabhutva bhoomini aakraminchukunna vaariki uchitamgaanuu, naamamaatrapu dharalaku kramabaddheekarinchaalani prabhutvam nirnayinchinatlu vaartalostunnaayi. idhi teevra garhaneeyame kaaka prabhutvam tana adhikaaraalanu durviniyogaparachadame avutundi. pedala musugulo vividhapaartiila naayakulaku, vaari taabedaarlaku prajala aastini dochipettadam tappa maremi kaadu. nijamgaa prajala kosame ayithe, chattalapai gowravamtho prabhutva sthalaalanu aakraminchukunna vaariki sahakarinchadamemiti? adhe samayamlo vela roopaayalu addelu chellistuu, adde illallo nivasistunna pedala matemiti? ante prabhutvam okarini – vaari kharmaniki vaarini vadilesi, chattavyatirekulaku andagaa nilavadamegaa! ippatike uchita kaanukalu vanti nirarthaka, anutpaadaka pathakaala dwara prajaadhanaanni vruthaa chestunnaru. mukhyamgaa raajakeeyanaayakulu, vividha sthaayillooni prajaapratinidhulu, vaari anucharaganaalaku tappa saamaanya pedha prajalaku aasinchinantagaa prayojanam dakkadam ledu. p

In [11]:
def split_sentences(text):
    sentences = re.split(r'(?<=[.!?।])\s+', text)
    return [s.strip() for s in sentences if s.strip()]

In [12]:
roman_sentences = split_sentences(sample["translit"])
telugu_sentences = split_sentences(sample["text"])

print("Roman sentences:", len(roman_sentences))
print("Telugu sentences:", len(telugu_sentences))

Roman sentences: 10
Telugu sentences: 10


In [13]:
for i, sentence in enumerate(roman_sentences):
    print(f"R{i}: {sentence}")

print("\n" + "=" * 80)

for i, sentence in enumerate(telugu_sentences):
    print(f"T{i}: {sentence}")

R0: rashtramlo prabhutva bhoomini aakraminchukunna vaariki uchitamgaanuu, naamamaatrapu dharalaku kramabaddheekarinchaalani prabhutvam nirnayinchinatlu vaartalostunnaayi.
R1: idhi teevra garhaneeyame kaaka prabhutvam tana adhikaaraalanu durviniyogaparachadame avutundi.
R2: pedala musugulo vividhapaartiila naayakulaku, vaari taabedaarlaku prajala aastini dochipettadam tappa maremi kaadu.
R3: nijamgaa prajala kosame ayithe, chattalapai gowravamtho prabhutva sthalaalanu aakraminchukunna vaariki sahakarinchadamemiti?
R4: adhe samayamlo vela roopaayalu addelu chellistuu, adde illallo nivasistunna pedala matemiti?
R5: ante prabhutvam okarini – vaari kharmaniki vaarini vadilesi, chattavyatirekulaku andagaa nilavadamegaa!
R6: ippatike uchita kaanukalu vanti nirarthaka, anutpaadaka pathakaala dwara prajaadhanaanni vruthaa chestunnaru.
R7: mukhyamgaa raajakeeyanaayakulu, vividha sthaayillooni prajaapratinidhulu, vaari anucharaganaalaku tappa saamaanya pedha prajalaku aasinchinantagaa prayojanam 

# Step -4: Data Cleaning

In [14]:
df = df[['translit', 'text']].copy()
df['translit'] = df['translit'].str.strip()
df['text'] = df['text'].str.strip()

In [15]:
print('Shape of Data Before = ', df.shape)
df = df.drop_duplicates(subset=['translit', 'text']).reset_index(drop=True)
print('Shape of Data After removing Duplicates = ', df.shape)

Shape of Data Before =  (47415, 2)
Shape of Data After removing Duplicates =  (46751, 2)


# Step -5: Feature Engineering

In [16]:
df['roman_words'] = df['translit'].str.split().str.len()
df['telugu_words'] = df['text'].str.split().str.len()

print(df[['roman_words', 'telugu_words']].describe())

        roman_words  telugu_words
count  46751.000000  46751.000000
mean     373.126393    373.122650
std      299.097326    299.093501
min        1.000000      1.000000
25%      113.000000    113.000000
50%      359.000000    359.000000
75%      530.000000    530.000000
max     6188.000000   6188.000000


In [17]:
def check_sentence_alignment(df):
    matching = 0
    mismatching = 0
    
    mismatch_examples = []
    
    for idx, row in df.iterrows():
        roman_telugu_sentences = split_sentences(row['translit'])
        telugu_sentences = split_sentences(row['text'])
        
        if len(roman_telugu_sentences) == len(telugu_sentences):
            matching += 1
        else:
            mismatching += 1
            mismatch_examples.append({
                "index" : idx,
                "roman_telugu_count": len(roman_telugu_sentences),
                "telugu_count": len(telugu_sentences)
            })
    return matching, mismatching, mismatch_examples

In [18]:
matching, mismatching, mismatch_examples = check_sentence_alignment(df)

print('Total paragraphs = ', len(df))
print('Matching Paragraphs = ', matching)
print('Mismatching Paragraphs = ', mismatching)
print('Match Percentage: ', matching / len(df) * 100)

Total paragraphs =  46751
Matching Paragraphs =  46745
Mismatching Paragraphs =  6
Match Percentage:  99.98716604992407


In [19]:
print(mismatch_examples)
for mismatch_record in mismatch_examples:
    print(df.iloc[mismatch_record['index']])

[{'index': 1990, 'roman_telugu_count': 122, 'telugu_count': 121}, {'index': 6943, 'roman_telugu_count': 64, 'telugu_count': 63}, {'index': 10472, 'roman_telugu_count': 110, 'telugu_count': 109}, {'index': 16882, 'roman_telugu_count': 72, 'telugu_count': 71}, {'index': 26951, 'roman_telugu_count': 75, 'telugu_count': 74}, {'index': 38120, 'roman_telugu_count': 121, 'telugu_count': 120}]
translit        "please! kaadanaku. aame maanasika sthitini dr...
text            "ప్లీజ్! కాదనకు. ఆమె మానసిక స్థితిని దృష్టిలో ...
roman_words                                                   701
telugu_words                                                  700
Name: 1990, dtype: object
translit        doosuku vachina bulletla sudulatho mukhyamantr...
text            దూసుకు వచ్చిన బుల్లెట్ల సుడులతో ముఖ్యమంత్రి శర...
roman_words                                                   353
telugu_words                                                  352
Name: 6943, dtype: object
translit        "vundundu bheem

In [20]:
# Lets create a new dataframe for sentences 
roman_sentences = []
telugu_sentences = []

for idx, row in df.iterrows():

    roman_s = split_sentences(row["translit"])
    telugu_s = split_sentences(row["text"])

    if len(roman_s) != len(telugu_s):
        continue

    roman_sentences.extend(roman_s)
    telugu_sentences.extend(telugu_s)

In [21]:
sentence_df = pd.DataFrame({
    "roman_telugu": roman_sentences,
    "telugu": telugu_sentences
})

print("Number of sentence pairs:", len(sentence_df))
print("Shape:", sentence_df.shape)

Number of sentence pairs: 2408549
Shape: (2408549, 2)


In [22]:
display(
    sentence_df.sample(20, random_state=42)
)

,roman_telugu,telugu
1204873,""" meernaaku kanipinchi rendu nelalu daatindanu...",""" మీర్నాకు కనిపించి రెండు నెలలు దాటిందనుకుంటాన..."
1215207,"""kakapothe aa fancy drus kaaryakramamlo raajak...","""కాకపోతే ఆ ఫ్యాన్సీ డ్రస్ కార్యక్రమంలో రాజకీయన..."
1535515,veellandaripai adhikari ratnakar gaari kokkari...,వీళ్ళందరిపై అధికారి రత్నాకర్ గారి కొక్కరికే తె...
596424,"""raatrayyaaka vastaanu.","""రాత్రయ్యాక వస్తాను."
1008567,"""enduku navvaaru?","""ఎందుకు నవ్వారు?"
480190,appudappudu ee dongalu 'pramukhula' illallo ch...,అప్పుడప్పుడు ఈ దొంగలు 'ప్రముఖుల' ఇళ్ళల్లో చోరీ...
901715,virigipoyina erragaju mukkani aa guntalo pades...,విరిగిపోయిన ఎర్రగాజు ముక్కని ఆ గుంటలో పడేసి మట...
460774,"""ivannee nenu aalochincha ledanukunnava?","""ఇవన్నీ నేను ఆలోచించ లేదనుకున్నావా?"
685555,kurcheelo valipoyi vunna tulasini chudagaane a...,కుర్చీలో వాలిపోయి వున్న తులసిని చూడగానే ఆందోళన...
1346179,gundello agni parvataalu ragulutunnaayi.,గుండెల్లో అగ్ని పర్వతాలు రగులుతున్నాయి.


In [23]:
print("Duplicate pairs:",
      sentence_df.duplicated(
          subset=["roman_telugu", "telugu"]
      ).sum())

Duplicate pairs: 178245


In [24]:
print("Empty Roman:",
      (sentence_df["roman_telugu"].str.strip() == "").sum())

print("Empty Telugu:",
      (sentence_df["telugu"].str.strip() == "").sum())

Empty Roman: 0
Empty Telugu: 0


In [25]:
sentence_df_clean = sentence_df.copy()

sentence_df_clean["roman_telugu"] = (
    sentence_df_clean["roman_telugu"]
    .str.strip()
)

sentence_df_clean["telugu"] = (
    sentence_df_clean["telugu"]
    .str.strip()
)

In [26]:
duplicate_count = sentence_df_clean.duplicated(
    subset=["roman_telugu", "telugu"]
).sum()

total = len(sentence_df_clean)

print("Total sentence pairs:", total)
print("Duplicate pairs:", duplicate_count)
print("Duplicate percentage:", duplicate_count / total * 100)

Total sentence pairs: 2408549
Duplicate pairs: 178245
Duplicate percentage: 7.4005137533012615


In [27]:
duplicates = sentence_df_clean[
    sentence_df_clean.duplicated(
        subset=["roman_telugu", "telugu"],
        keep=False
    )
].sort_values(["roman_telugu", "telugu"])

display(duplicates.head(30))

,roman_telugu,telugu
2129006,,
2200929,,
2232977,,
2295964,,
2331436,,
2384494,,
2401056,,
2110605,,
2257352,,
2271810,,


In [28]:
df_clean = df.drop(index=[mismatch['index'] for mismatch in mismatch_examples]).reset_index(drop=True)

print("Original paragraphs:", len(df))
print("Removed mismatched paragraphs:", len(mismatch_examples))
print("Clean paragraphs:", len(df_clean))

Original paragraphs: 46751
Removed mismatched paragraphs: 6
Clean paragraphs: 46745


In [29]:
RANDOM_STATE = 42

In [30]:
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42

train_df, temp_df = train_test_split(
    df_clean,
    test_size=0.20,
    random_state=RANDOM_STATE
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=RANDOM_STATE
)

print("Train paragraphs:", len(train_df))
print("Validation paragraphs:", len(val_df))
print("Test paragraphs:", len(test_df))

Train paragraphs: 37396
Validation paragraphs: 4674
Test paragraphs: 4675


In [31]:
def create_sentence_pairs(paragraph_df):
    roman_sentences = []
    telugu_sentences = []

    skipped = 0

    for _, row in paragraph_df.iterrows():

        roman_s = split_sentences(row["translit"])
        telugu_s = split_sentences(row["text"])

        if len(roman_s) != len(telugu_s):
            skipped += 1
            continue

        roman_sentences.extend(roman_s)
        telugu_sentences.extend(telugu_s)

    sentence_df = pd.DataFrame({
        "roman_telugu": roman_sentences,
        "telugu": telugu_sentences
    })

    return sentence_df, skipped

In [32]:
train_sentences, train_skipped = create_sentence_pairs(train_df)

val_sentences, val_skipped = create_sentence_pairs(val_df)

test_sentences, test_skipped = create_sentence_pairs(test_df)

In [33]:
print("Train sentence pairs:", len(train_sentences))
print("Validation sentence pairs:", len(val_sentences))
print("Test sentence pairs:", len(test_sentences))

print("\nSkipped paragraphs:")
print("Train:", train_skipped)
print("Validation:", val_skipped)
print("Test:", test_skipped)

Train sentence pairs: 1928446
Validation sentence pairs: 237389
Test sentence pairs: 242714

Skipped paragraphs:
Train: 0
Validation: 0
Test: 0


In [34]:
print(train_sentences.shape)
print(val_sentences.shape)
print(test_sentences.shape)

(1928446, 2)
(237389, 2)
(242714, 2)


In [35]:
display(train_sentences.sample(10, random_state=42))

,roman_telugu,telugu
1871987,maa taatagaaru naa chinnappudu daanni naaku ba...,మా తాతగారు నా చిన్నప్పుడు దాన్ని నాకు బహూకరించ...
1204734,"""detective anveshi lekunda, akiyothoni ee test...","""డిటెక్టివ్ అన్వేషి లేకుండా, అకియోతోని ఈ టెస్ట..."
557327,"""mee valla naa sneham, navallana?","""మీ వల్ల నా స్నేహం, నావల్లనా?"
1030570,"chaalaamandi.""\r\n \r\n ""andulo meeku telisina...","చాలామంది.""\r\n \r\n ""అందులో మీకు తెలిసిన వాళ్ళ..."
1767959,ee vyavahaaram antaa teleeni sitaranemo yintik...,ఈ వ్యవహారం అంతా తెలీని సితారనేమో యింటికెళ్ళిపో...
1265460,na...,న...
991937,"""jaagratta!","""జాగ్రత్త!"
113615,nenu magavaadinani prapanchaaniki bujuvayindi.,నేను మగవాడినని ప్రపంచానికి బుజువయింది.
290675,"cheekatlo ettugaa, needala madhya nallagaa poy...","చీకట్లో ఎత్తుగా, నీడలా మధ్య నల్లగా పోయిన తన పర..."
1386889,samaadhaanamgaa bhaaryanu rendu chetulloki tee...,సమాధానంగా భార్యను రెండు చేతుల్లోకి తీసుకున్నాడు.


In [36]:
print("Train duplicates:",
      train_sentences.duplicated(
          subset=["roman_telugu", "telugu"]
      ).sum())

print("Validation duplicates:",
      val_sentences.duplicated(
          subset=["roman_telugu", "telugu"]
      ).sum())

print("Test duplicates:",
      test_sentences.duplicated(
          subset=["roman_telugu", "telugu"]
      ).sum())

Train duplicates: 135256
Validation duplicates: 10941
Test duplicates: 11236


In [37]:
train_pairs = set(
    zip(
        train_sentences["roman_telugu"],
        train_sentences["telugu"]
    )
)

val_pairs = set(
    zip(
        val_sentences["roman_telugu"],
        val_sentences["telugu"]
    )
)

test_pairs = set(
    zip(
        test_sentences["roman_telugu"],
        test_sentences["telugu"]
    )
)

print("Unique train pairs:", len(train_pairs))
print("Unique validation pairs:", len(val_pairs))
print("Unique test pairs:", len(test_pairs))

Unique train pairs: 1793190
Unique validation pairs: 226448
Unique test pairs: 231478


In [38]:
train_test_overlap = train_pairs.intersection(test_pairs)
train_val_overlap = train_pairs.intersection(val_pairs)
val_test_overlap = val_pairs.intersection(test_pairs)

print("Train ↔ Test overlap:", len(train_test_overlap))
print("Train ↔ Validation overlap:", len(train_val_overlap))
print("Validation ↔ Test overlap:", len(val_test_overlap))

Train ↔ Test overlap: 9863
Train ↔ Validation overlap: 9961
Validation ↔ Test overlap: 3256


In [39]:
roman_target_counts = (
    train_sentences
    .groupby("roman_telugu")["telugu"]
    .nunique()
)

conflicting_roman = roman_target_counts[
    roman_target_counts > 1
]

print("Roman sentences with multiple Telugu targets:",
      len(conflicting_roman))

Roman sentences with multiple Telugu targets: 1575


In [40]:
for roman_sentence in conflicting_roman.index[:20]:

    print("=" * 80)
    print("ROMAN:")
    print(roman_sentence)

    print("\nTELUGU TARGETS:")
    print(
        train_sentences[
            train_sentences["roman_telugu"] == roman_sentence
        ]["telugu"]
        .unique()
    )

ROMAN:
" amma!

TELUGU TARGETS:
<ArrowStringArray>
['" అమ్మా!', '" అమ్మ!']
Length: 2, dtype: str
ROMAN:
"aa !

TELUGU TARGETS:
<ArrowStringArray>
['"ఆ !', '"ఆఁ !']
Length: 2, dtype: str
ROMAN:
"aa ....

TELUGU TARGETS:
<ArrowStringArray>
['"ఆ ....', '"ఆఁ ....']
Length: 2, dtype: str
ROMAN:
"aa aa !

TELUGU TARGETS:
<ArrowStringArray>
['"ఆ ఆ !', '"ఆఁ ఆఁ !']
Length: 2, dtype: str
ROMAN:
"aa meera?

TELUGU TARGETS:
<ArrowStringArray>
['"ఆ మీరా?', '"ఆఁ మీరా?']
Length: 2, dtype: str
ROMAN:
"aa nene!

TELUGU TARGETS:
<ArrowStringArray>
['"ఆ నేనే!', '"ఆఁ నేనే!']
Length: 2, dtype: str
ROMAN:
"aa!

TELUGU TARGETS:
<ArrowStringArray>
['"ఆఁ!', '"ఆ!']
Length: 2, dtype: str
ROMAN:
"aa!" andi.

TELUGU TARGETS:
<ArrowStringArray>
['"ఆ!" అంది.', '"ఆఁ!" అంది.']
Length: 2, dtype: str
ROMAN:
"aa.

TELUGU TARGETS:
<ArrowStringArray>
['"ఆ.', '"ఆఁ.']
Length: 2, dtype: str
ROMAN:
"aa...

TELUGU TARGETS:
<ArrowStringArray>
['"ఆ...', '"ఆఁ...']
Length: 2, dtype: str
ROMAN:
"aa...!

TELUGU TARGETS:
<ArrowStringA

In [41]:
telugu_source_counts = (
    train_sentences
    .groupby("telugu")["roman_telugu"]
    .nunique()
)

conflicting_telugu = telugu_source_counts[
    telugu_source_counts > 1
]

print(
    "Telugu sentences with multiple Roman representations:",
    len(conflicting_telugu)
)

Telugu sentences with multiple Roman representations: 50


In [42]:
train_pair_set = set(
    zip(
        train_sentences["roman_telugu"],
        train_sentences["telugu"]
    )
)

val_pairs = list(
    zip(
        val_sentences["roman_telugu"],
        val_sentences["telugu"]
    )
)

test_pairs = list(
    zip(
        test_sentences["roman_telugu"],
        test_sentences["telugu"]
    )
)

In [43]:
val_clean = val_sentences[
    ~val_sentences.apply(
        lambda row: (row["roman_telugu"], row["telugu"]) in train_pair_set,
        axis=1
    )
].reset_index(drop=True)

test_clean = test_sentences[
    ~test_sentences.apply(
        lambda row: (row["roman_telugu"], row["telugu"]) in train_pair_set,
        axis=1
    )
].reset_index(drop=True)

In [44]:
print("Original validation:", len(val_sentences))
print("Clean validation:", len(val_clean))

print("Original test:", len(test_sentences))
print("Clean test:", len(test_clean))

Original validation: 237389
Clean validation: 220459
Original test: 242714
Clean test: 225264


In [45]:
train_sentences["roman_length"] = (
    train_sentences["roman_telugu"].str.split().str.len()
)

train_sentences["telugu_length"] = (
    train_sentences["telugu"].str.split().str.len()
)

In [46]:
print(train_sentences[
    ["roman_length", "telugu_length"]
].describe(percentiles=[0.50, 0.90, 0.95, 0.99]))

       roman_length  telugu_length
count  1.928446e+06   1.928446e+06
mean   7.249267e+00   7.249198e+00
std    5.753874e+00   5.753843e+00
min    1.000000e+00   1.000000e+00
50%    6.000000e+00   6.000000e+00
90%    1.400000e+01   1.400000e+01
95%    1.700000e+01   1.700000e+01
99%    2.700000e+01   2.700000e+01
max    5.060000e+02   5.060000e+02


In [47]:
print(
    "Roman 99th percentile:",
    train_sentences["roman_length"].quantile(0.99)
)

print(
    "Telugu 99th percentile:",
    train_sentences["telugu_length"].quantile(0.99)
)

Roman 99th percentile: 27.0
Telugu 99th percentile: 27.0


In [48]:
MAX_LENGTH = 40

roman_long = (
    train_sentences["roman_length"] > MAX_LENGTH
)

telugu_long = (
    train_sentences["telugu_length"] > MAX_LENGTH
)

too_long = roman_long | telugu_long

print("Total training pairs:", len(train_sentences))
print("Pairs > MAX_LENGTH:", too_long.sum())
print(
    "Percentage removed:",
    too_long.mean() * 100
)

Total training pairs: 1928446
Pairs > MAX_LENGTH: 3941
Percentage removed: 0.20436143921063904


In [49]:
for name, data in [
    ("Train", train_sentences),
    ("Validation", val_clean),
    ("Test", test_clean)
]:
    roman_len = data["roman_telugu"].str.split().str.len()
    telugu_len = data["telugu"].str.split().str.len()

    too_long = (roman_len > MAX_LENGTH) | (telugu_len > MAX_LENGTH)

    print(
        f"{name}: {too_long.sum()} / {len(data)} "
        f"({too_long.mean() * 100:.2f}%)"
    )

Train: 3941 / 1928446 (0.20%)
Validation: 424 / 220459 (0.19%)
Test: 456 / 225264 (0.20%)


##### Confirming the max_length and dataset is correct

In [50]:
MAX_LENGTH = 40

def filter_by_length(df, max_length=40):
    df = df.copy()

    roman_length = df["roman_telugu"].str.split().str.len()
    telugu_length = df["telugu"].str.split().str.len()

    mask = (
        (roman_length <= max_length) &
        (telugu_length <= max_length)
    )

    return df.loc[mask].reset_index(drop=True)

In [51]:
train_final = filter_by_length(train_sentences, MAX_LENGTH)
val_final = filter_by_length(val_clean, MAX_LENGTH)
test_final = filter_by_length(test_clean, MAX_LENGTH)

In [52]:
print("Train:", train_final.shape)
print("Validation:", val_final.shape)
print("Test:", test_final.shape)

Train: (1924505, 4)
Validation: (220035, 2)
Test: (224808, 2)


In [53]:
for name, data in [
    ("Train", train_final),
    ("Validation", val_final),
    ("Test", test_final)
]:
    
    roman_max = data["roman_telugu"].str.split().str.len().max()
    telugu_max = data["telugu"].str.split().str.len().max()

    print(
        name,
        "Roman max =", roman_max,
        "Telugu max =", telugu_max
    )

Train Roman max = 40 Telugu max = 40
Validation Roman max = 40 Telugu max = 40
Test Roman max = 40 Telugu max = 40


In [54]:
display(
    train_final.sample(10, random_state=42)
)

,roman_telugu,telugu,roman_length,telugu_length
94791,1911 british prabhutvam konni grandhaalanu nis...,1911 బ్రిటీష్ ప్రభుత్వం కొన్ని గ్రంథాలను నిషేధ...,6,6
1680619,"""ee roju eeyana mana cheef gest!","""ఈ రోజు ఈయన మన చీఫ్ గెస్ట్!",6,6
809018,'post' anna keka vini ramanujangaru parugettuk...,'పోస్ట్' అన్న కేక విని రామానుజంగారు పరుగెత్తుక...,6,6
297218,saadhusatvapu sodaratvapu\r\nswaadutattvam jay...,సాధుసత్వపు సోదరత్వపు\r\nస్వాదుతత్త్వం జయిస్తుం...,10,10
193761,"ivaala lechina vaela manchidi"" andi paarvati, ...","ఇవాళ లేచిన వేళ మంచిది"" అంది పార్వతి, వణుకుతున్...",8,8
224867,"aa tarvaata nyoo ginia, jimbabwelatho talapada...","ఆ తర్వాత న్యూ గినియా, జింబాబ్వేలతో తలపడనుంది.",6,6
659972,hengarla ninda vunna cheerale oo rendu vandala...,హేంగర్ల నిండా వున్న చీరలే ఓ రెండు వందల దాకా వు...,9,9
714451,deeniki etuvanti mugimpu labhistundo chuudaala...,దీనికి ఎటువంటి ముగింపు లభిస్తుందో చూడాలన్నారు.,5,5
82931,asalu tanaki lav leter anedi ela rastaro teliy...,అసలు తనకి లవ్ లెటర్ అనేది ఎలా రాస్తారో తేలియదు.,8,8
299747,aa funksionku povan kalyan vastada?,ఆ ఫంక్షన్కు పవన్ కల్యాణ్ వస్తాడా?,5,5


##### Final Data is prepared

In [55]:
train_final = train_final[["roman_telugu", "telugu"]].copy()
val_final = val_final[["roman_telugu", "telugu"]].copy()
test_final = test_final[["roman_telugu", "telugu"]].copy()

In [56]:
print(train_final.shape)
print(val_final.shape)
print(test_final.shape)

print(train_final.columns.tolist())

(1924505, 2)
(220035, 2)
(224808, 2)
['roman_telugu', 'telugu']


# Step -6: Tokenization

In [57]:
def tokenize(text):
    text = re.sub(
        r'([.,!?;:"\'()\[\]{}\-—…])',
        r' \1 ',
        text
    )

    return text.split()

In [58]:
print(tokenize("nenu intiki vellanu."))
print(tokenize("నేను ఇంటికి వెళ్ళాను."))

['nenu', 'intiki', 'vellanu', '.']
['నేను', 'ఇంటికి', 'వెళ్ళాను', '.']


In [59]:
print(tokenize('"amma!'))

['"', 'amma', '!']


In [60]:
from collections import Counter

roman_counter = Counter()
telugu_counter = Counter()

for text in train_final["roman_telugu"]:
    roman_counter.update(tokenize(text))

for text in train_final["telugu"]:
    telugu_counter.update(tokenize(text))

print("Roman vocabulary:", len(roman_counter))
print("Telugu vocabulary:", len(telugu_counter))

print("\nRoman:")
print(roman_counter.most_common(30))

print("\nTelugu:")
print(telugu_counter.most_common(30))

Roman vocabulary: 1169272
Telugu vocabulary: 1230935

Roman:
[('.', 2467317), ('"', 782079), (',', 425073), ('!', 226891), ('?', 208131), ('aa', 143671), ('ee', 107329), ('-', 97754), ("'", 71222), ('annadu', 59580), ('ani', 58993), ('kuudaa', 58906), ('aame', 58569), ('naa', 53897), ('nenu', 53532), ('tana', 48532), ('oka', 41260), ('*', 40478), ('oo', 36574), ('andi', 36242), ('nee', 35783), ('mee', 32366), ('naaku', 32287), ('vundi', 32058), ('ledu', 31901), ('chala', 30362), ('nunchi', 29601), ('adhi', 28969), ('athani', 26723), ('cony', 25826)]

Telugu:
[('.', 2467317), ('"', 782079), (',', 425073), ('!', 226891), ('?', 208131), ('ఆ', 142519), ('ఈ', 107303), ('-', 97754), ("'", 71222), ('అన్నాడు', 59537), ('అని', 58980), ('కూడా', 58904), ('ఆమె', 58256), ('నా', 53694), ('నేను', 53527), ('తన', 48530), ('ఒక', 41239), ('*', 40478), ('అంది', 36202), ('ఓ', 34943), ('నీ', 33440), ('నాకు', 32278), ('మీ', 31172), ('లేదు', 30573), ('చాలా', 30357), ('వుంది', 29596), ('అది', 28891), ('నుంచి',

In [61]:
print(
    "Roman tokens appearing once:",
    sum(1 for count in roman_counter.values() if count == 1)
)

print(
    "Telugu tokens appearing once:",
    sum(1 for count in telugu_counter.values() if count == 1)
)

Roman tokens appearing once: 738768
Telugu tokens appearing once: 788582


In [62]:
def clean_sentence(text):
    # Remove standalone * characters
    text = re.sub(r'(?<!\S)\*(?!\S)', ' ', text)

    # Normalize multiple spaces
    text = re.sub(r'\s+', ' ', text)

    return text.strip()

In [63]:
print(clean_sentence('* * *'))
print(clean_sentence('nenu intiki vellanu.'))


nenu intiki vellanu.


In [64]:
for data in [train_final, val_final, test_final]:
    data["roman_telugu"] = data["roman_telugu"].apply(clean_sentence)
    data["telugu"] = data["telugu"].apply(clean_sentence)

In [65]:
def remove_empty_rows(data):
    return data[
        (data["roman_telugu"].str.strip() != "") &
        (data["telugu"].str.strip() != "")
    ].reset_index(drop=True)

train_final = remove_empty_rows(train_final)
val_final = remove_empty_rows(val_final)
test_final = remove_empty_rows(test_final)

In [66]:
print(train_final.shape)
print(val_final.shape)
print(test_final.shape)

(1923914, 2)
(220035, 2)
(224807, 2)


In [67]:
def vocab_stats(counter, thresholds=(1, 2, 3, 5, 10, 20)):
    total_tokens = sum(counter.values())

    for threshold in thresholds:
        vocab_size = sum(
            1 for count in counter.values()
            if count >= threshold
        )

        covered_tokens = sum(
            count for count in counter.values()
            if count >= threshold
        )

        coverage = covered_tokens / total_tokens * 100

        print(
            f"min_freq={threshold:2d} | "
            f"vocab={vocab_size:8d} | "
            f"coverage={coverage:6.2f}%"
        )

In [68]:
print("ROMAN")
vocab_stats(roman_counter)

print("\nTELUGU")
vocab_stats(telugu_counter)

ROMAN
min_freq= 1 | vocab= 1169272 | coverage=100.00%
min_freq= 2 | vocab=  430504 | coverage= 95.94%
min_freq= 3 | vocab=  278119 | coverage= 94.26%
min_freq= 5 | vocab=  176259 | coverage= 92.37%
min_freq=10 | vocab=  100320 | coverage= 89.66%
min_freq=20 | vocab=   58586 | coverage= 86.56%

TELUGU
min_freq= 1 | vocab= 1230935 | coverage=100.00%
min_freq= 2 | vocab=  442353 | coverage= 95.66%
min_freq= 3 | vocab=  284196 | coverage= 93.92%
min_freq= 5 | vocab=  179477 | coverage= 91.98%
min_freq=10 | vocab=  101977 | coverage= 89.22%
min_freq=20 | vocab=   59344 | coverage= 86.05%


# Step -7: Modelling

In [69]:
MIN_FREQ = 10
MAX_LENGTH = 40

In [70]:
from collections import Counter

PAD_TOKEN = "<PAD>"
UNK_TOKEN = "<UNK>"
SOS_TOKEN = "<SOS>"
EOS_TOKEN = "<EOS>"


class Vocabulary:

    def __init__(self, min_freq=10):
        self.min_freq = min_freq

        self.token_to_idx = {
            PAD_TOKEN: 0,
            UNK_TOKEN: 1,
            SOS_TOKEN: 2,
            EOS_TOKEN: 3,
        }

        self.idx_to_token = {
            0: PAD_TOKEN,
            1: UNK_TOKEN,
            2: SOS_TOKEN,
            3: EOS_TOKEN,
        }

    def build(self, counter):

        for token, count in counter.items():

            if count >= self.min_freq:

                idx = len(self.token_to_idx)

                self.token_to_idx[token] = idx
                self.idx_to_token[idx] = token

    def __len__(self):
        return len(self.token_to_idx)

    def numericalize(self, tokens):

        return [
            self.token_to_idx.get(token, self.token_to_idx[UNK_TOKEN])
            for token in tokens
        ]

    def decode(self, indices):

        return [
            self.idx_to_token.get(idx, UNK_TOKEN)
            for idx in indices
        ]

In [71]:
MIN_FREQ = 10

roman_vocab = Vocabulary(MIN_FREQ)
telugu_vocab = Vocabulary(MIN_FREQ)

roman_vocab.build(roman_counter)
telugu_vocab.build(telugu_counter)

In [72]:
print("Roman vocab size:", len(roman_vocab))
print("Telugu vocab size:", len(telugu_vocab))

Roman vocab size: 100324
Telugu vocab size: 101981


In [73]:
text = "nenu intiki vellanu."

In [74]:
tokens = tokenize(text)

print(tokens)

['nenu', 'intiki', 'vellanu', '.']


In [75]:
ids = roman_vocab.numericalize(tokens)

print(ids)

[335, 2670, 5786, 13]


In [76]:
tokens = tokenize("xyzabc nenu")
print(tokens)

ids = roman_vocab.numericalize(tokens)
print(ids)

['xyzabc', 'nenu']
[1, 335]


In [77]:
text = "నేను ఇంటికి వెళ్ళాను."

tokens = tokenize(text)

print(tokens)

ids = telugu_vocab.numericalize(tokens)

print(ids)

['నేను', 'ఇంటికి', 'వెళ్ళాను', '.']
[336, 2676, 27597, 13]


In [78]:
def numericalize_source(text, vocab):
    tokens = tokenize(text)
    return vocab.numericalize(tokens)


def numericalize_target(text, vocab):
    tokens = tokenize(text)

    token_ids = vocab.numericalize(tokens)

    decoder_input = [vocab.token_to_idx[SOS_TOKEN]] + token_ids
    decoder_target = token_ids + [vocab.token_to_idx[EOS_TOKEN]]

    return decoder_input, decoder_target

In [79]:
source = "nenu intiki vellanu."
target = "నేను ఇంటికి వెళ్ళాను."

source_ids = numericalize_source(
    source,
    roman_vocab
)

decoder_input, decoder_target = numericalize_target(
    target,
    telugu_vocab
)

print("Source:", source_ids)
print("Decoder input:", decoder_input)
print("Decoder target:", decoder_target)

Source: [335, 2670, 5786, 13]
Decoder input: [2, 336, 2676, 27597, 13]
Decoder target: [336, 2676, 27597, 13, 3]


In [80]:
train_final["roman_token_length"] = (
    train_final["roman_telugu"].apply(
        lambda x: len(tokenize(x))
    )
)

train_final["telugu_token_length"] = (
    train_final["telugu"].apply(
        lambda x: len(tokenize(x))
    )
)

In [81]:
print(
    train_final[
        ["roman_token_length", "telugu_token_length"]
    ].describe(
        percentiles=[0.50, 0.90, 0.95, 0.99]
    )
)

       roman_token_length  telugu_token_length
count        1.923914e+06         1.923914e+06
mean         9.426357e+00         9.426357e+00
std          6.544073e+00         6.544070e+00
min          1.000000e+00         1.000000e+00
50%          8.000000e+00         8.000000e+00
90%          1.700000e+01         1.700000e+01
95%          2.200000e+01         2.200000e+01
99%          3.400000e+01         3.400000e+01
max          1.690000e+02         1.690000e+02


In [82]:
print(
    "Roman max:",
    train_final["roman_token_length"].max()
)

print(
    "Telugu max:",
    train_final["telugu_token_length"].max()
)

Roman max: 169
Telugu max: 169


In [83]:
MAX_TOKEN_LENGTH = 50

for name, data in [
    ("Train", train_final),
    ("Validation", val_final),
    ("Test", test_final)
]:
    roman_len = data["roman_telugu"].apply(
        lambda x: len(tokenize(x))
    )

    telugu_len = data["telugu"].apply(
        lambda x: len(tokenize(x))
    )

    too_long = (
        (roman_len > MAX_TOKEN_LENGTH) |
        (telugu_len > MAX_TOKEN_LENGTH)
    )

    print(
        f"{name}: "
        f"{too_long.sum()} / {len(data)} "
        f"({too_long.mean() * 100:.3f}%)"
    )

Train: 2002 / 1923914 (0.104%)
Validation: 206 / 220035 (0.094%)
Test: 231 / 224807 (0.103%)


In [84]:
def filter_by_token_length(data, max_length=50):
    data = data.copy()

    roman_len = data["roman_telugu"].apply(
        lambda x: len(tokenize(x))
    )

    telugu_len = data["telugu"].apply(
        lambda x: len(tokenize(x))
    )

    mask = (
        (roman_len <= max_length) &
        (telugu_len <= max_length)
    )

    return data.loc[mask].reset_index(drop=True)

In [85]:
MAX_TOKEN_LENGTH = 50

train_final = filter_by_token_length(
    train_final,
    MAX_TOKEN_LENGTH
)

val_final = filter_by_token_length(
    val_final,
    MAX_TOKEN_LENGTH
)

test_final = filter_by_token_length(
    test_final,
    MAX_TOKEN_LENGTH
)

In [86]:
print("Train:", train_final.shape)
print("Validation:", val_final.shape)
print("Test:", test_final.shape)

Train: (1921912, 4)
Validation: (219829, 2)
Test: (224576, 2)


In [87]:
train_final = train_final[
    ["roman_telugu", "telugu"]
].copy()

val_final = val_final[
    ["roman_telugu", "telugu"]
].copy()

test_final = test_final[
    ["roman_telugu", "telugu"]
].copy()

In [88]:
print(train_final.columns)
print(val_final.columns)
print(test_final.columns)

Index(['roman_telugu', 'telugu'], dtype='str')
Index(['roman_telugu', 'telugu'], dtype='str')
Index(['roman_telugu', 'telugu'], dtype='str')


In [89]:
print("Empty Roman:")
print(train_final["roman_telugu"].eq("").sum())

print("Empty Telugu:")
print(train_final["telugu"].eq("").sum())

Empty Roman:
0
Empty Telugu:
0


In [90]:
sample = train_final.iloc[100]

print("Roman:")
print(sample["roman_telugu"])

print("\nTelugu:")
print(sample["telugu"])

print("\nRoman tokens:")
print(tokenize(sample["roman_telugu"]))

print("\nTelugu tokens:")
print(tokenize(sample["telugu"]))

Roman:
‘helo inastagram..

Telugu:
‘హెలో ఇనస్టాగ్రమ్..

Roman tokens:
['‘helo', 'inastagram', '.', '.']

Telugu tokens:
['‘హెలో', 'ఇనస్టాగ్రమ్', '.', '.']


In [91]:
import re

def tokenize(text):
    text = re.sub(
        r'([.,!?;:"\'‘’“”()\[\]{}\-—…])',
        r' \1 ',
        text
    )

    return text.split()

In [92]:
print(tokenize("‘helo inastagram.."))
print(tokenize("‘హెలో ఇనస్టాగ్రమ్.."))
print(tokenize('"amma!'))
print(tokenize("'amma!"))

['‘', 'helo', 'inastagram', '.', '.']
['‘', 'హెలో', 'ఇనస్టాగ్రమ్', '.', '.']
['"', 'amma', '!']
["'", 'amma', '!']


In [93]:
train_final = train_final[
    ["roman_telugu", "telugu"]
].copy()

val_final = val_final[
    ["roman_telugu", "telugu"]
].copy()

test_final = test_final[
    ["roman_telugu", "telugu"]
].copy()

In [94]:
print(train_final.shape)
print(train_final.columns)

(1921912, 2)
Index(['roman_telugu', 'telugu'], dtype='str')


In [95]:
from collections import Counter

roman_counter = Counter()
telugu_counter = Counter()

for text in train_final["roman_telugu"]:
    roman_counter.update(tokenize(text))

for text in train_final["telugu"]:
    telugu_counter.update(tokenize(text))

In [96]:
print("Roman raw vocabulary:", len(roman_counter))
print("Telugu raw vocabulary:", len(telugu_counter))

print("\nRoman:")
print(roman_counter.most_common(30))

print("\nTelugu:")
print(telugu_counter.most_common(30))

Roman raw vocabulary: 1150368
Telugu raw vocabulary: 1211768

Roman:
[('.', 2439956), ('"', 769454), (',', 420510), ('!', 226003), ('?', 206169), ('aa', 143027), ('ee', 107087), ('-', 96303), ("'", 70338), ('annadu', 59409), ('ani', 58786), ('kuudaa', 58718), ('aame', 58441), ('naa', 53705), ('nenu', 53390), ('tana', 48413), ('oka', 41147), ('oo', 36466), ('andi', 36147), ('nee', 35508), ('naaku', 32129), ('mee', 32111), ('vundi', 31957), ('ledu', 31847), ('chala', 30258), ('nunchi', 29478), ('adhi', 28839), ('athani', 26668), ('cony', 25689), ('maa', 25613)]

Telugu:
[('.', 2439956), ('"', 769454), (',', 420510), ('!', 226003), ('?', 206169), ('ఆ', 141890), ('ఈ', 107059), ('-', 96303), ("'", 70338), ('అన్నాడు', 59366), ('అని', 58773), ('కూడా', 58716), ('ఆమె', 58128), ('నా', 53504), ('నేను', 53385), ('తన', 48411), ('ఒక', 41126), ('అంది', 36108), ('ఓ', 34862), ('నీ', 33178), ('నాకు', 32120), ('మీ', 30918), ('లేదు', 30521), ('చాలా', 30253), ('వుంది', 29508), ('అది', 28762), ('నుంచి', 278

In [97]:
print(
    "Roman tokens appearing once:",
    sum(1 for c in roman_counter.values() if c == 1)
)

print(
    "Telugu tokens appearing once:",
    sum(1 for c in telugu_counter.values() if c == 1)
)

Roman tokens appearing once: 725751
Telugu tokens appearing once: 775279


In [98]:
MIN_FREQ = 10

In [99]:
roman_vocab = Vocabulary(MIN_FREQ)
telugu_vocab = Vocabulary(MIN_FREQ)

roman_vocab.build(roman_counter)
telugu_vocab.build(telugu_counter)

In [100]:
print("Roman vocab size:", len(roman_vocab))
print("Telugu vocab size:", len(telugu_vocab))

Roman vocab size: 99932
Telugu vocab size: 101595


In [101]:
text = "‘helo inastagram.."

tokens = tokenize(text)

print(tokens)

ids = roman_vocab.numericalize(tokens)

print(ids)

['‘', 'helo', 'inastagram', '.', '.']
[506, 1, 1, 13, 13]


In [102]:
MAX_TOKEN_LENGTH = 50

def filter_by_token_length(data, max_length=50):
    data = data.copy()

    roman_len = data["roman_telugu"].apply(
        lambda x: len(tokenize(x))
    )

    telugu_len = data["telugu"].apply(
        lambda x: len(tokenize(x))
    )

    mask = (
        (roman_len <= max_length) &
        (telugu_len <= max_length)
    )

    return data.loc[mask].reset_index(drop=True)

In [103]:
train_final = filter_by_token_length(
    train_final,
    MAX_TOKEN_LENGTH
)

val_final = filter_by_token_length(
    val_final,
    MAX_TOKEN_LENGTH
)

test_final = filter_by_token_length(
    test_final,
    MAX_TOKEN_LENGTH
)

In [104]:
train_final = train_final[
    ["roman_telugu", "telugu"]
].copy()

val_final = val_final[
    ["roman_telugu", "telugu"]
].copy()

test_final = test_final[
    ["roman_telugu", "telugu"]
].copy()

#### Create Dataset

In [105]:
import torch
from torch.utils.data import Dataset


class RomanTeluguDataset(Dataset):

    def __init__(
        self,
        dataframe,
        roman_vocab,
        telugu_vocab,
        max_source_len=50,
        max_target_len=52
    ):
        self.dataframe = dataframe.reset_index(drop=True)

        self.roman_vocab = roman_vocab
        self.telugu_vocab = telugu_vocab

        self.max_source_len = max_source_len
        self.max_target_len = max_target_len

        self.pad_idx = roman_vocab.token_to_idx[PAD_TOKEN]
        self.sos_idx = telugu_vocab.token_to_idx[SOS_TOKEN]
        self.eos_idx = telugu_vocab.token_to_idx[EOS_TOKEN]

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):

        roman_text = self.dataframe.iloc[index]["roman_telugu"]
        telugu_text = self.dataframe.iloc[index]["telugu"]

        # 1. Tokenize
        roman_tokens = tokenize(roman_text)
        telugu_tokens = tokenize(telugu_text)

        # 2. Convert tokens → IDs
        source_ids = self.roman_vocab.numericalize(
            roman_tokens
        )

        target_ids = self.telugu_vocab.numericalize(
            telugu_tokens
        )

        # 3. Add SOS/EOS to target
        decoder_input = [
            self.sos_idx
        ] + target_ids

        decoder_target = (
            target_ids + [self.eos_idx]
        )

        # 4. Pad source
        source_ids = (
            source_ids
            + [self.pad_idx] *
            (self.max_source_len - len(source_ids))
        )

        # 5. Pad decoder input
        decoder_input = (
            decoder_input
            + [self.pad_idx] *
            (self.max_target_len - len(decoder_input))
        )

        # 6. Pad decoder target
        decoder_target = (
            decoder_target
            + [self.pad_idx] *
            (self.max_target_len - len(decoder_target))
        )

        return {
            "source": torch.tensor(
                source_ids,
                dtype=torch.long
            ),

            "decoder_input": torch.tensor(
                decoder_input,
                dtype=torch.long
            ),

            "decoder_target": torch.tensor(
                decoder_target,
                dtype=torch.long
            )
        }

In [106]:
train_dataset = RomanTeluguDataset(
    train_final,
    roman_vocab,
    telugu_vocab
)

val_dataset = RomanTeluguDataset(
    val_final,
    roman_vocab,
    telugu_vocab
)

test_dataset = RomanTeluguDataset(
    test_final,
    roman_vocab,
    telugu_vocab
)

In [107]:
print("Train:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Test:", len(test_dataset))

Train: 1921845
Validation: 219825
Test: 224568


##### Check with a sample

In [108]:
sample = train_dataset[0]

print(sample)

{'source': tensor([ 4,  5,  6,  7,  8,  9, 10, 11, 12, 13,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0]), 'decoder_input': tensor([ 2,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0]), 'decoder_target': tensor([ 4,  5,  6,  7,  8,  9, 10, 11, 12, 13,  3,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0])}


In [109]:
print("Source shape:",
      sample["source"].shape)

print("Decoder input shape:",
      sample["decoder_input"].shape)

print("Decoder target shape:",
      sample["decoder_target"].shape)

Source shape: torch.Size([50])
Decoder input shape: torch.Size([52])
Decoder target shape: torch.Size([52])


In [110]:
source_decoded = roman_vocab.decode(
    sample["source"].tolist()
)

target_decoded = telugu_vocab.decode(
    sample["decoder_target"].tolist()
)

print("SOURCE:")
print(source_decoded)

print("\nTARGET:")
print(target_decoded)

SOURCE:
['sankraanti', 'panduga', 'ante', 'ventane', 'gurtukochedi', 'gaalipataalu', ',', 'kodi', 'pandelu', '.', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>']

TARGET:
['సంక్రాంతి', 'పండుగ', 'అంటే', 'వెంటనే', 'గుర్తుకొచ్చేది', 'గాలిపటాలు', ',', 'కోడి', 'పందేలు', '.', '<EOS>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>']


#### Create Data Loader

In [111]:
from torch.utils.data import DataLoader

BATCH_SIZE = 64

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

In [112]:
batch = next(iter(train_loader))

# print(batch.keys())

print("Source:",
      batch["source"].shape)

print("Decoder input:",
      batch["decoder_input"].shape)

print("Decoder target:",
      batch["decoder_target"].shape)

Source: torch.Size([64, 50])
Decoder input: torch.Size([64, 52])
Decoder target: torch.Size([64, 52])


In [113]:
print(batch["source"][0])
print(batch["decoder_input"][0])
print(batch["decoder_target"][0])

tensor([   74, 16853,   960,     1,   368,  6733,    13,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0])
tensor([    2,    74, 16943,   957,     1,   370,  6745,    13,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0])
tensor([   74, 16943,   957,     1,   370,  6745,    13,     3,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     

#### Create the Model:

##### Encoder

In [114]:
import torch
import torch.nn as nn


class Encoder(nn.Module):

    def __init__(
        self,
        input_vocab_size,
        embedding_dim,
        hidden_dim
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            num_embeddings=input_vocab_size,
            embedding_dim=embedding_dim,
            padding_idx=0
        )

        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            batch_first=True
        )

    def forward(self, source):

        # source:
        # [batch, source_length]

        embedded = self.embedding(source)

        # embedded:
        # [batch, source_length, embedding_dim]

        outputs, (hidden, cell) = self.lstm(
            embedded
        )

        return outputs, hidden, cell

In [115]:
EMBEDDING_DIM = 128
HIDDEN_DIM = 256

encoder = Encoder(
    input_vocab_size=len(roman_vocab),
    embedding_dim=EMBEDDING_DIM,
    hidden_dim=HIDDEN_DIM
)

In [116]:
print(encoder)

Encoder(
  (embedding): Embedding(99932, 128, padding_idx=0)
  (lstm): LSTM(128, 256, batch_first=True)
)


In [117]:
batch = next(iter(train_loader))

source = batch["source"]

outputs, hidden, cell = encoder(source)

In [118]:
batch = next(iter(train_loader))

source = batch["source"]

outputs, hidden, cell = encoder(source)

In [119]:
print("Source:", source.shape)
print("Encoder outputs:", outputs.shape)
print("Hidden:", hidden.shape)
print("Cell:", cell.shape)

Source: torch.Size([64, 50])
Encoder outputs: torch.Size([64, 50, 256])
Hidden: torch.Size([1, 64, 256])
Cell: torch.Size([1, 64, 256])


##### Decoder

In [120]:
class Decoder(nn.Module):

    def __init__(
        self,
        output_vocab_size,
        embedding_dim,
        hidden_dim
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            num_embeddings=output_vocab_size,
            embedding_dim=embedding_dim,
            padding_idx=0
        )

        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            batch_first=True
        )

        self.fc_out = nn.Linear(
            hidden_dim,
            output_vocab_size
        )

    def forward(
        self,
        decoder_input,
        hidden,
        cell
    ):

        # decoder_input:
        # [batch, target_length]

        embedded = self.embedding(
            decoder_input
        )

        # embedded:
        # [batch, target_length, embedding_dim]

        outputs, (hidden, cell) = self.lstm(
            embedded,
            (hidden, cell)
        )

        # outputs:
        # [batch, target_length, hidden_dim]

        predictions = self.fc_out(outputs)

        # predictions:
        # [batch, target_length, output_vocab_size]

        return predictions, hidden, cell

In [121]:
decoder = Decoder(
    output_vocab_size=len(telugu_vocab),
    embedding_dim=EMBEDDING_DIM,
    hidden_dim=HIDDEN_DIM
)

In [122]:
print(decoder)

Decoder(
  (embedding): Embedding(101595, 128, padding_idx=0)
  (lstm): LSTM(128, 256, batch_first=True)
  (fc_out): Linear(in_features=256, out_features=101595, bias=True)
)


In [123]:
batch = next(iter(train_loader))

source = batch["source"]
decoder_input = batch["decoder_input"]

encoder_outputs, hidden, cell = encoder(source)

predictions, hidden2, cell2 = decoder(
    decoder_input,
    hidden,
    cell
)

In [124]:
print("Decoder input:",
      decoder_input.shape)

print("Predictions:",
      predictions.shape)

print("Hidden:",
      hidden2.shape)

print("Cell:",
      cell2.shape)

Decoder input: torch.Size([64, 52])
Predictions: torch.Size([64, 52, 101595])
Hidden: torch.Size([1, 64, 256])
Cell: torch.Size([1, 64, 256])


In [125]:
class Seq2Seq(nn.Module):

    def __init__(self, encoder, decoder):
        super().__init__()

        self.encoder = encoder
        self.decoder = decoder

    def forward(self, source, decoder_input):

        # Encoder
        encoder_outputs, hidden, cell = self.encoder(source)

        # Decoder
        predictions, hidden, cell = self.decoder(
            decoder_input,
            hidden,
            cell
        )

        return predictions

In [126]:
model = Seq2Seq(
    encoder,
    decoder
)

In [127]:
print(model)

Seq2Seq(
  (encoder): Encoder(
    (embedding): Embedding(99932, 128, padding_idx=0)
    (lstm): LSTM(128, 256, batch_first=True)
  )
  (decoder): Decoder(
    (embedding): Embedding(101595, 128, padding_idx=0)
    (lstm): LSTM(128, 256, batch_first=True)
    (fc_out): Linear(in_features=256, out_features=101595, bias=True)
  )
)


In [128]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(device)

cuda


In [129]:
model = model.to(device)

In [130]:
batch = next(iter(train_loader))

source = batch["source"].to(device)
decoder_input = batch["decoder_input"].to(device)
decoder_target = batch["decoder_target"].to(device)

In [131]:
print(source.device)

cuda:0


In [132]:
predictions = model(
    source,
    decoder_input
)

In [133]:
print(predictions.shape)

torch.Size([64, 52, 101595])


In [134]:
criterion = nn.CrossEntropyLoss(
    ignore_index=0
)

In [135]:
output_dim = predictions.shape[-1]

predictions = predictions.reshape(
    -1,
    output_dim
)

decoder_target = decoder_target.reshape(
    -1
)

In [136]:
loss = criterion(
    predictions,
    decoder_target
)

print("Loss:", loss.item())

Loss: 11.512811660766602


In [137]:
torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

tensor(0.)

In [138]:
criterion = nn.CrossEntropyLoss(ignore_index=0)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

In [139]:
model.train()

batch = next(iter(train_loader))

source = batch["source"].to(device)
decoder_input = batch["decoder_input"].to(device)
decoder_target = batch["decoder_target"].to(device)

optimizer.zero_grad()

predictions = model(
    source,
    decoder_input
)

output_dim = predictions.shape[-1]

predictions = predictions.reshape(
    -1,
    output_dim
)

decoder_target = decoder_target.reshape(
    -1
)

loss = criterion(
    predictions,
    decoder_target
)

print("Loss before backward:", loss.item())

loss.backward()

torch.nn.utils.clip_grad_norm_(
    model.parameters(),
    max_norm=1.0
)

optimizer.step()

print("One training step completed!")

Loss before backward: 11.512789726257324
One training step completed!


In [140]:
total_params = sum(
    p.numel()
    for p in model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print(
    "Total parameters:",
    total_params
)

print(
    "Trainable parameters:",
    trainable_params
)

Total parameters: 52695899
Trainable parameters: 52695899


In [141]:
def evaluate(
    model,
    loader,
    criterion,
    device
):
    model.eval()

    total_loss = 0

    with torch.no_grad():

        for batch in loader:

            source = batch["source"].to(device)
            decoder_input = batch["decoder_input"].to(device)
            decoder_target = batch["decoder_target"].to(device)

            predictions = model(
                source,
                decoder_input
            )

            output_dim = predictions.shape[-1]

            predictions = predictions.reshape(
                -1,
                output_dim
            )

            decoder_target = decoder_target.reshape(
                -1
            )

            loss = criterion(
                predictions,
                decoder_target
            )

            total_loss += loss.item()

    return total_loss / len(loader)

In [142]:
def train_one_epoch(
    model,
    loader,
    optimizer,
    criterion,
    device,
    clip=1.0
):
    model.train()

    total_loss = 0

    for batch in loader:

        source = batch["source"].to(device)
        decoder_input = batch["decoder_input"].to(device)
        decoder_target = batch["decoder_target"].to(device)

        optimizer.zero_grad()

        predictions = model(
            source,
            decoder_input
        )

        output_dim = predictions.shape[-1]

        predictions = predictions.reshape(
            -1,
            output_dim
        )

        decoder_target = decoder_target.reshape(
            -1
        )

        loss = criterion(
            predictions,
            decoder_target
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            clip
        )

        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

In [143]:
import time

model.train()

start_time = time.time()

for batch_idx, batch in enumerate(train_loader):

    source = batch["source"].to(device)
    decoder_input = batch["decoder_input"].to(device)
    decoder_target = batch["decoder_target"].to(device)

    optimizer.zero_grad()

    predictions = model(
        source,
        decoder_input
    )

    output_dim = predictions.shape[-1]

    predictions = predictions.reshape(
        -1,
        output_dim
    )

    decoder_target = decoder_target.reshape(
        -1
    )

    loss = criterion(
        predictions,
        decoder_target
    )

    loss.backward()

    torch.nn.utils.clip_grad_norm_(
        model.parameters(),
        1.0
    )

    optimizer.step()

    if batch_idx == 99:
        break

elapsed = time.time() - start_time

print(
    f"100 batches took {elapsed:.2f} seconds"
)

print(
    f"Average per batch: {elapsed / 100:.3f} seconds"
)

100 batches took 14.36 seconds
Average per batch: 0.144 seconds


In [144]:
train_loss = train_one_epoch(
    model,
    train_loader,
    optimizer,
    criterion,
    device
)

print("Train Loss:", train_loss)

val_loss = evaluate(
    model,
    val_loader,
    criterion,
    device
)

print("Validation Loss:", val_loss)

Train Loss: 3.735057680057183
Validation Loss: 2.612251841813716


In [158]:
def _seq2seq_predict(self, text, max_len=MAX_LENGTH):
    """Greedily translate one Roman Telugu sentence into Telugu."""
    self.eval()
    model_device = next(self.parameters()).device

    source_tokens = tokenize(text)
    if not source_tokens:
        return ""

    source_ids = roman_vocab.numericalize(source_tokens)
    source = torch.tensor([source_ids], dtype=torch.long, device=model_device)

    with torch.no_grad():
        _, hidden, cell = self.encoder(source)
        current_token = torch.tensor(
            [[telugu_vocab.token_to_idx[SOS_TOKEN]]],
            dtype=torch.long,
            device=model_device,
        )
        generated_ids = []

        for _ in range(max_len):
            logits, hidden, cell = self.decoder(current_token, hidden, cell)
            next_id = logits[:, -1, :].argmax(dim=-1)
            token_id = int(next_id.item())

            if token_id == telugu_vocab.token_to_idx[EOS_TOKEN]:
                break
            if token_id not in (
                telugu_vocab.token_to_idx[PAD_TOKEN],
                telugu_vocab.token_to_idx[SOS_TOKEN],
            ):
                generated_ids.append(token_id)

            current_token = next_id.unsqueeze(1)

    return " ".join(telugu_vocab.decode(generated_ids))


# Attach the method to the already-trained model class without recreating the model.
Seq2Seq.predict = _seq2seq_predict

print(model.predict("telidu"))

" అంది .


In [159]:
from pathlib import Path
import json

artifact_dir = Path("/home/cs24m118/practice/projects/p1/roman_telugu_api/artifacts")
artifact_dir.mkdir(parents=True, exist_ok=True)

model_config = {
    "embedding_dim": EMBEDDING_DIM,
    "hidden_dim": HIDDEN_DIM,
    "max_length": MAX_LENGTH,
    "input_vocab_size": len(roman_vocab),
    "output_vocab_size": len(telugu_vocab),
}

checkpoint_path = artifact_dir / "seq2seq.pt"
torch.save(
    {
        "state_dict": {
            key: value.detach().cpu()
            for key, value in model.state_dict().items()
        },
        "config": model_config,
    },
    checkpoint_path,
)

vocab_data = {
    "roman": [roman_vocab.idx_to_token[i] for i in range(len(roman_vocab))],
    "telugu": [telugu_vocab.idx_to_token[i] for i in range(len(telugu_vocab))],
    "special_tokens": {
        "pad": PAD_TOKEN,
        "unk": UNK_TOKEN,
        "sos": SOS_TOKEN,
        "eos": EOS_TOKEN,
    },
}
with (artifact_dir / "vocab.json").open("w", encoding="utf-8") as f:
    json.dump(vocab_data, f, ensure_ascii=False)

print(f"Saved weights: {checkpoint_path} ({checkpoint_path.stat().st_size / 1024 / 1024:.2f} MiB)")
print(f"Saved vocab: {artifact_dir / 'vocab.json'}")
print(f"Config: {model_config}")

Saved weights: /home/cs24m118/practice/projects/p1/roman_telugu_api/artifacts/seq2seq.pt (201.02 MiB)
Saved vocab: /home/cs24m118/practice/projects/p1/roman_telugu_api/artifacts/vocab.json
Config: {'embedding_dim': 128, 'hidden_dim': 256, 'max_length': 40, 'input_vocab_size': 99932, 'output_vocab_size': 101595}


In [ ]:
import zipfile

bundle_path = artifact_dir / "roman_telugu_model.zip"
with zipfile.ZipFile(bundle_path, "w", compression=zipfile.ZIP_DEFLATED) as bundle:
    bundle.write(artifact_dir / "seq2seq.pt", arcname="seq2seq.pt")
    bundle.write(artifact_dir / "vocab.json", arcname="vocab.json")

print(f"Download bundle: {bundle_path} ({bundle_path.stat().st_size / 1024 / 1024:.2f} MiB)")